In [2]:
print('Loading AL_functions')       

Loading AL_functions


In [225]:
#moving average function with the added option to reflect data at the ends to retain a full set of data points 
def moving_average(a, n=3, reflect = True):
    if reflect :
        N = int(n/2)
        s = a.size
        a = np.hstack((a[N:0:-1],a,a[s-2:s-N-2:-1]))
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

In [208]:
def smoother(ds,varname,smoothed):
    # Over short intervals of time land surface phenomena are often relatively constant, but remote sensing images are often noisey,
    # producing a confused time series.
    # This function reduces the noise by estimating the true value of the point based on it,and the preceeding and suceeding values.
    # 
    # --- Using the in-built interp function to apply two linear interpolations to be used as smoothers.
    # --- the challenge is keeping data at the start and end dates; for simplicity I am retaining the original values but that is not ideal.
    # --- Equivalent to estimating the value of a point from a linear regression through the point and its nearest partners, with additional weight on the target point. 
    #     (If the intervales between the points is constant then it is exactly eqivalent)
    # I think this is just equivalent to a moving average value with increased weight on the central observation, 
    # but it is a convernient implementation!

    # improvement would be to generalise to other dimensions, e.g., x and y.
    
    if varname  == None: varname  = 'wofs_ann_freq'
    if smoothed == None: smoothed =  varname+'_sm'

    even_times = ds.time[0::2]
    odd_times  = ds.time[1::2]

    vodd = (ds.sel(time=even_times)[varname].interp(time=odd_times,method='linear') + (ds.sel(time=odd_times)[varname] * 2) )/3
    veve = (ds.sel(time=odd_times)[varname].interp(time=even_times,method='linear') + (ds.sel(time=even_times)[varname] * 2) )/3

    ds[smoothed] = ds[varname] + np.nan     # --- sets up a variable ---
    ds[smoothed].loc[even_times,:,:] = veve
    ds[smoothed].loc[odd_times,:,: ] = vodd
    # --- restore the end values (ugly approach but it will do for now)
    ds[smoothed] = xr.where(np.isnan(ds[smoothed]),ds[varname],ds[smoothed])
    return(ds)

In [4]:
def plot_wq_timeseries(summary,measure='tss',placename='Lake_Baringo',verbose=False):
    
    dimension = measure+'_measure'
    parameters ={'tss'  : {'yminmax' :[0,150]},
                 'chla' : {'yminmax' :[0,200 ]},}
    
    yminmax = parameters[measure]['yminmax']

    #---- use a two-step interpolation to fit a smooth trend curve 
    #     piecewise_linreg replaces each point with the estimated value of the point based on it, and the two points either side of it in time.
    #     although I made this technique up as common sense, I've since seen that it has a name... somewhere...
    # ---- remove nodata dates so that the 1-D quadratic spline does not fail
    
    xy = piecewise_linreg(\
        np.arange(0,summary.time.size),\
        summary[measure].loc[placename,:].clip(yminmax[0],yminmax[1]).median(dim=(dimension)),\
        keepnans=True)[0]

    xlist = [] ; ylist = []
    for i in range(0,summary.time.size):
        if xy[1][i] >=0 :
            xlist.append(summary.time[i].values)
            ylist.append(xy[1][i])

    xlist = np.asarray(xlist)
    ylist = np.asarray(ylist)
    
    q_interp   = sp.interpolate.interp1d(xlist,ylist,kind='quadratic',fill_value='extrapolate')

    #make a denser time series to allow representation of the median with a smooth curve
    t_new      = pd.date_range(summary.time.min().values, summary.time.max().values, freq='2M')  

    #--- for the plot title, access more informtion on the site:
    places = set_spacetime_domain()
    titlestring = "--- "+str(placename)+" : "+measure+" --- \n"+\
    str(places[placename]['desc'])+\
    "  longitude: "+str(places[placename]['xyt']['x'])+\
    "  latitude:  "+str(places[placename]['xyt']['y'])
    
    plt.figure(figsize=[12, 6])
    summary[measure].loc[placename,:].clip(0,150).plot.line\
            (hue=dimension,linestyle='dotted',ylim=(0,yminmax[1]*1.5),\
             xlim=(summary.time[0],summary.time[summary.time.size-1]))

    summary[measure].loc[placename,:].clip(0,150)\
        .median(dim=(dimension))\
        .plot.line\
        ('bo')
    plt.plot(t_new,q_interp(t_new),'b',linestyle="dashed")
    plt.title(titlestring)
    plt.show()

In [ ]:
def msi_oli_regression(ds,wft,verbose= True, test = True) :
    if test : print('wft = ',wft,'\n total cells in dataset = ',(ds.x.size * ds.y.size * ds.time.size))

    #first, a structure to store regression coefficients for now (only supports a simple regression...)
    oli_from_msi = {
    #                'oli01':  {'msi_band':'msi01','params':([1,0]),'r_params':([0,0]),'hls_params':([1.005,0.000209])},
                    'oli02_agm':  {'msi_band':'msi02_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([1.020,0.00447])},
                    'oli03_agm' : {'msi_band':'msi03_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([0.994,0.00109])},
                    'oli04_agm' : {'msi_band':'msi04_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([1.017,0.00104])},
    #                'oli05_agm' : {'msi_band':'msi08_agm','params':([1,0])},'r_params':([0,0]),'hls_params':([0.999,0.00025]),
    #                'oli06_agm' : {'msi_band':'msi11_agm','params':([1,0])},'r_params':([0,0]),'hls_params':([0.999,0.000124]),
                    'oli07_agm' : {'msi_band':'msi12_agm','params':([1,0]),'r_params':([0,0]),'hls_params':([1.003,0.00119])}     
                    }

    #make a dictionary stack of the relevant arrays if they exist in the dataset
    #the variables are set to zero for areas that are outside of the water areas,using the wft variable.
    varstack = {}
    for oli_varname in oli_from_msi.keys():
        if oli_varname in ds.data_vars :
            msi_varname = oli_from_msi[oli_varname]['msi_band']
            if msi_varname in ds.data_vars:
                varstack[oli_varname] = ds[oli_varname].where(ds.wofs_ann_freq >= wft,0).where(ds[oli_varname]>0,0)
                varstack[msi_varname] = ds[msi_varname].where(ds.wofs_ann_freq >= wft,0).where(ds[msi_varname]>0,0)
    
    if np.size(list(varstack.keys())) == 0:
        print('no variables found in the dataset for regression')
   
    #below, we reduce the size of the regression problem by randomly selecting a smaller number  of cells, after reshaping
    total_cells = np.size(ds[list(varstack.keys())[0]])
    mask        = np.ones((total_cells),dtype='int8')   #'mask' will be an index

    #convert to 1Xn vector and set the mask to nan if either of the inputs are nan
    for varname in varstack.keys():
        varstack[varname] = np.asarray(varstack[varname]).reshape(total_cells)
        mask = np.all((mask,varstack[varname]),0)                           #any non-zero value, including a nan, will return True

    #at this point the mask should be those places where msi and oli data are both present, and the water frequency isabove hte thold.
    #apply the mask to reduce to observations where all values are present
    for varname in varstack.keys():
        varstack[varname] = varstack[varname][mask>0]    #nan values are not > 0

    #reduce to the desired number of samples through a random selection
    value_cells  = np.size(varstack[varname])      #this is how many valid observations we have
    target_cells = 100000                          #this is how many valid observations we want...
    rmask        = (np.random.rand(value_cells))<(target_cells/value_cells) 
    for varname in varstack.keys():
        varstack[varname] = varstack[varname][rmask]

    if test: 
        print('\n random selection to reduec to about ',target_cells)
        print('\n size of ',varname,'  :   ',np.size(varstack[varname]))
        print('\n','-------------------------------')

    #Only implemented a simple linear regression here, could try multivariate (the function supports it) but 
    #don't have time to set it up now, and over-fitting beceomes an issue. 
    #loop through the oli and msi variables, calculate the regression and the inverse regression

    for oli_var in oli_from_msi.keys():               
        msi_var = oli_from_msi[oli_var]['msi_band']
        y = varstack[oli_var]
        A = np.ones(len(y))    #correstponds to the offset
        A = np.vstack((A,varstack[msi_var]))
        (oli_from_msi[oli_var]['params'][1],oli_from_msi[oli_var]['params'][0]),b,c,d = \
            sp.linalg.lstsq(A.T,y)   #returns intercept, slope, other..

        # reverse regression, msi from oli
        y = varstack[msi_var]
        A = np.ones(len(y))           #correstponds to the offset
        A = np.vstack((A,varstack[oli_var]))
        (oli_from_msi[oli_var]['r_params'][1],oli_from_msi[oli_var]['r_params'][0]),b,c,d = \
            sp.linalg.lstsq(A.T,y)   #returns intercept, slope, other..
    
    #the dictionary oli_from_msi now contains the regression slope and intercept 

    #Plot the results
    if test or verbose :
        i = 0
        plt.figure(figsize=[12, 3])
        for oli_var in list(oli_from_msi.keys()):
            i = i + 1
            plt.subplot(int('14' + str(i)))
            #oli_var                  = 'oli03_agm'
            msi_var                  = oli_from_msi[oli_var]['msi_band']
            slope,intercept          = oli_from_msi[oli_var]['params']
            slope_hls,intercept_hls = oli_from_msi[oli_var]['hls_params'] 
            x = varstack[msi_var]
            y = varstack[oli_var]

            # plot the results
            #plt.hist2d(x,y,bins=20,cmap='grey')
            plt.plot(x,y,'yo')
            plt.plot(x,slope*x+intercept,'b')
            plt.plot(x,slope_hls*x+intercept_hls, '--k')

            #, x, x*slope + intercept '--k',x, (x*slope_hls + intercept_hls) 'b') #'--k'=black dashed line, 'yo' = yellow circle marker
            plt.xlabel(msi_var)
            plt.ylabel(oli_var)

            plt.xlim(np.min(x)*0.8, np.quantile(x,.999)*1.5)
            plt.ylim(np.min(y)*0.8, np.quantile(y,.999)*1.5)
        plt.show()
    return(oli_from_msi)

In [6]:
from scipy.interpolate import interp1d
def piecewise_linreg(x,y,keepnans=True,test=False):
    #from two series, representing paired  x,y values, this function estimates the value of each y point with a linear regresion of the points either side of it, and itself
    #It alows one to fit a trend line to the series that just approximates the data point.   
    # the process needs to ignore nans, but the returned sequence must be consistent in x values, so here I remove all nans, do the fits, replace nans
    #this won't work if x is datetime64 because the variables won't stack.
    
    a      = np.vstack((x,y,y))
    aprime = np.copy(a)
        
    newx = np.argwhere(a.sum(0)>=0)     #no nans in these columns
    xlen = newx.size                
    b    = np.zeros(a.shape[0]*xlen).reshape(a.shape[0],xlen)*np.nan
    for i in np.arange(0,xlen):
        b[:,i] = a[:,newx[i][0]]    
    a = b  

    centre_weighted = True
    for i in range(0,len(a[0])-2) : 
        if centre_weighted:
            y = np.append(a[1][i:i+3],a[1][i+1])
            x = np.append(a[0][i:i+3],a[0][i+1])
            A = np.vstack((np.ones(4)*1.,x))
        else:
            y = a[1][i:i+3]
            x = a[0][i:i+3]
            A = np.vstack((np.ones(3)*1.,x))
                
        (intercept,slope) = sp.linalg.lstsq(A.T,y)[0]
        def f(x) : 
            return (x * slope + intercept).round(3)
        a[2,i+1] = f(a[0][i+1])
        if test : print(f(a[0][i+1]))
        if i == 0  : 
            a[2,i]   = f(a[0][i])
        if i == len(a[0])-3 : 
            if test : print('i = ',i)
            a[2,i+2] = f(a[0][i+2])

    if keepnans:
        #bring back the complete series, with nans:
        for i in range(0,xlen) :
            aprime[:,newx[i][0]] = a[:,i]
        a = aprime
            
    bias,sd = sum(a[2]-a[1]),((a[2]-a[1])**2).sum()**0.5
    if test : print('bias = ',bias.round(3),'sd = ',sd.round(3))
    #return(a[[0,2],:],bias,sd)
    return([a[0,:],a[2,:]],bias,sd)

### Wofs summary, statistics give a robust estimate of water area during and between years. 
Version 1. aims to implement uncertainty and gap-filling

#### Average Area of Water (AAW):
* This is the sum of the water frequency values, times the cell area
* Limitations of the AAW:
  * Just one number
  * Not a good indicator of the area of water to expect
  * Affected by surrounding flood plains etc., which have large area but low frequency, will push up the average.

#### Percentile water area (PWA, e.g. PWA50):
* if the percentile is 50%, then this gives the area of water to be expected 50% of the time. This is more robust than the AAW in the same way that the median is more robust than the mean.
* Advantages
  * The percentile can be set at any value, e.g., PWA90 would be the area of places full 90% of the time.
  * Not sensitive to the inclusion, or exclusion, of surrounding areas such as floodplains. Areas that have frequency less than the nominated percentile dont appear in the calculation.
* Limitations of the PWA:
  * Valid only on a single waterbody, not, for instance a string of waterholes that fill and empty independently. The assumption in the PWA is that if a pixel of, say, freq  0.5 is filled, then any pixel with freq > 0.5 will also be filled. In words: '1: deeper areas will have higher freq; 2:deeper areas will will always be wet if shallower areas are wet'
  * PWA only uses the observations that are above a certain threshold. When looking at low water, there are less observations of water, with more potential for cloud or data gaps to impact. PWA curves are less reliable when looking at low water because there are fewer pixels to count.

#### Total Count Water Area (TCWA)
* TCWA is an attempt to estimate the average water area based on all the observations, as a global estimate
* Calculated as 
  [sum_of_water_count  / sum_of_clear_count] ..... _for the entire area_
* I.E.,  the frequency of water,  _for the waterbody_
* For this estimate, I define _the waterbody_ to be any area that has wofs_freq > 0 _for any year_. This gives the cell count for estimation of the area of surface water, the 'areal' n.

* The variance of the mean can be estimated as _npq_. Exactly how to calculate this needs checking.

* This seemed to be a very simple and elegant aproach, but would only work if each count was weighted inversely with the probability of getting a positive count at that cell. I think that would take us back to the AAW calculation. Results are not helpful so I have discontinuted.



## Some better water area stats? (bias is the killer; and over cloudy waterbodies, that's the problem!)

Foundation.
Assuming no detection error (perfect algorithm) (best possible case)
- p is the probability of positive, q = 1-p. We don't know p but can estimate from all-time wofs
- e(x), a single observation, is p*1 + q*0 = [p, q][1, 0].T  = p
- sigma(x), is p*(1-p)^2 + q*(p)^2 = pq (once you work it through)
For a series of N obs at a pixel:
- mean(N) is expected to be p
- sigma2(N) = (pq/N) is the variance of the mean, and the mean is the frequency.

From the above, we can say, in a best case situation, how many observations would be needed to be able to say, for a given observation interval: 
"we are very confident that this pixel was not water more than X% of the observation period." 
Waters where we could not say this, should be regarded as permanent.

Using two conditions we should be able to put a bound on the area of permanent water.
"we are very confident that this pixel was water more than X% of the obs period
Waters where we can say this, should be regarded as permanent.

Specifically, the 2-sigma condition is that p + 2*sqrt(pq/N) < X, or p - 2*sqrt(pq/N) < X if p>X. 

The condition for equality is N = 4pq/(x-p)^2; (note that goes infinite as x and p converge, because no amount of obs will prove they are not the same)
So, for a given X, p and N, we can say wether we should count that pixel as 'yes', or 'no'.

Given that observations are imperfect, this places a lower bound on the number of observations required to eliminate a pixel from the class X.
Similarly, we can simply place a bound on p using the 2-sigma range.

Variants:
- the number of observations needed to get to a given level of accuracy / as a confidence measure : actualobs/required obs


### sample variance, based on the current wofs time series
We can make a similar calculation based on the sample mean and variance:
(W/N)*(1-W/N)/(N-1) 

### estimated area of water 
- the estimated area of water from the mean of the frequencies
- the variance of the estimate based on the summed variance of each of the contributing pixels

### and in practice ...

- Unfortunately, the actual uncertainty (bias) in the frequency - in particular underestimation over large waterbodies - is actually greater than this uncertainty. Consequently some improvements to wofs are essential to get to an accurate estimate of permanent water.


In [ ]:
# --- moved to _WQ_functions.py
def water_analysis_DEPRECATED(ds,
                   water_frequency_threshold= 0.5,
                   wofs_varname             = 'wofs_ann_freq',
                   permanent_water_threshold= 0.875,
                   sigma_coefficient        = 1.2,
                   verbose                  = True,
                   test                     = True):
    
    if not wofs_varname in ('wofs_ann_freq','wofs_fill_freq') : 
        print ('INVALID VARIABLE NAME, defaulting to wofs_ann_freq')
        wofs_varname = 'wofs_ann_freq'

    # --- standard deviation of the annual frequency at each pixel - should really be dividing by n-1 but then I would need to change SC ---
    ds['wofs_ann_freq_sigma'] = ((ds.wofs_ann_freq * (1 - ds.wofs_ann_freq)) / ds.wofs_ann_clearcount)**0.5
    ds['wofs_pw_threshold']   = (-1 * ds.wofs_ann_freq_sigma * sigma_coefficient) + permanent_water_threshold  #--- threshold varies with p and n 
    ds['wofs_ann_pwater']     = xr.where(ds[wofs_varname]> ds.pw_threshold,ds[wofs_varname],0)
    ds['wofs_ann_water']      = xr.where(ds[wofs_varname]> water_frequency_threshold_high,ds[wofs_varname],0)
    return(ds)
    

In [ ]:
def wofs_summary(summary,ds,water_frequency_threshold,
                 wofs_varname='wofs_ann_freq',
                 PWT=0.875,
                 SC=1.2,
                 rfname = 'water_area_results.txt',
                 verbose=True,test=True):
    if verbose:     print('Calculating WOfS summary...')
    if not wofs_varname in ('wofs_ann_freq','wofs_fill_freq') : 
        print ('INVALID VARIABLE NAME, defaulting to wofs_ann_freq')
        wofs_varname = 'wofs_ann_freq'
    # --- some parameters ---
    placename       = str(summary.placename.values)
    PW_threshold    = PWT
    sigma_coefficient = SC
        
    # --- run basic water area analysis for permanent water
    ds = water_analysis(ds,
                        water_frequency_threshold=water_frequency_threshold,
                        wofs_varname=wofs_varname,
                        permanent_water_threshold = PWT,
                        sigma_coefficient=SC,
                        verbose=verbose,test=test)
    
    # --- area summary steps --- 
    cell_area = ((ds.x[1]-ds.x[0]) * (ds.y[1]-ds.y[0])) / 1000000
    Reference_period  = ['2000','2019']
    Assessment_period = ['2020','2024']
    reference_times   = summary.time.loc[Reference_period[0]:Reference_period[1]]
    assessment_times   = summary.time.loc[Assessment_period[0]:Assessment_period[1]]
    
    summary['w50'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.50).count(dim=('x','y')) * cell_area)
    summary['w05'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.95).count(dim=('x','y')) * cell_area)
    summary['w95'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.05).count(dim=('x','y')) * cell_area)
    summary['w90'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.10).count(dim=('x','y')) * cell_area)
    summary['w80'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.20).count(dim=('x','y')) * cell_area)
    
    summary['w10'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.90).count(dim=('x','y')) * cell_area)
    summary['w20'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.80).count(dim=('x','y')) * cell_area)
    summary['w30'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.70).count(dim=('x','y')) * cell_area)
    summary['w40'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.60).count(dim=('x','y')) * cell_area)
    summary['avf'] = (ds[wofs_varname].sum(dim=('x','y')) * cell_area) 

    #The last is what is used in the vectorize waterbodies notebook. It is the average area of water. 
    #The average is not a good indicator of what to expect because floodplains, and geography/geometry, skew the distribution.
    #It is also more affected by missing data than, say W50. 
    #In clear-sky areas with floodplains, the average will be higher than w50
    #In cloud-affected areas without floodplains, the average will be below w50
    
    #    summary['wofs_np'] = (ds.wofs_ann_wetcount * ds.wofs_ann_freq).where(ds.wofs_ann_freq>=0.8).median(dim=('x','y'))
    
    #overall water area - any place observed as water during the observation period.
    water_area = ds.wofs_ann_wetcount.max(dim='time')[:,:].where(ds.wofs_ann_freq.max(dim='time')[:,:] > .0).count() * cell_area
    if test : print('maximum ever water area calculated as : ',water_area,' Total study area is ',ds.x.size * ds.y.size * cell_area)
       
    # --- save time series summary --- 
    # --- the min/max values are highly conservative since they assume that all pixels vary in the same direction
    summary['wofs_ann_pwater']     = \
            ds[wofs_varname].where(ds[wofs_varname]                                > ds.wofs_pw_threshold).count(dim=('x','y')) * cell_area
    summary['wofs_ann_pwater_min'] = \
            ds[wofs_varname].where(ds[wofs_varname] - (ds.wofs_ann_freq_sigma * 2) > PW_threshold).count(dim=('x','y')) * cell_area
    summary['wofs_ann_pwater_max'] = \
            ds[wofs_varname].where(ds[wofs_varname] + (ds.wofs_ann_freq_sigma * 2) > PW_threshold).count(dim=('x','y')) * cell_area

    # --- the mean frequency is also a measure of the average water area
    summary['wofs_ann_freq_mean'] = ds.wofs_ann_freq.mean(dim=('y','x'))
    summary['wofs_ann_freq_sd'  ] = (ds.wofs_ann_freq.var(dim=('y','x'))**0.5)

    # --- Change statistics:
    reference_pw_area  = summary.wofs_ann_pwater.loc[Reference_period[0]:Reference_period[1]].median(dim='time').values
    assessment_pw_area = summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]].median(dim='time').values
    pw_change_percent = max(abs((100 * (assessment_pw_area / reference_pw_area - 1.0))),
                            0.000000000000001) # --- to prevent problems with infinities if change is really zero--#

    if test : print(reference_pw_area,assessment_pw_area,pw_change_percent)
    
    # --- round these values:
    if reference_pw_area > 0.0001 :
        reference_pw_area  = np.round(reference_pw_area,3-int((np.log10(reference_pw_area))))
        if assessment_pw_area >0.00001:
            assessment_pw_area  = np.round(assessment_pw_area,3-int((np.log10(assessment_pw_area))))
        else : assessment_pw_area = np.round(0.0)
        pw_change_percent  = np.round(pw_change_percent,3-int((np.log10(abs(pw_change_percent)))))
    else:
        reference_pwater_area = np.round(0.0)
    if not assessment_pw_area > 0.00001: assessment_pw_area = np.round(0.0)
    if test : 
        print('types:',type(assessment_pw_area),type(reference_pw_area))
    
    
    assessment_uq_area = summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]].quantile(.75,dim='time').values
    assessment_lq_area = summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]].quantile(.25,dim='time').values
    reference_uq_area  = summary.wofs_ann_pwater.loc[Reference_period[0] :Reference_period[1]] .quantile(.75,dim='time').values
    reference_lq_area  = summary.wofs_ann_pwater.loc[Reference_period[0] :Reference_period[1]] .quantile(.25,dim='time').values
    
    # ----------------------------------------------------    
    # --- Robust regression on the annual areas for the reference and assessment period ---
    #slope,intercept,low_slope,high_slope = sp.stats.theilslopes(y,x,method='joint')
    x = summary.time 
    y = summary.wofs_ann_pwater 
    slope,intercept,low_slope,high_slope = sp.stats.theilslopes(y,x,method='joint')
    
    if low_slope < 0 and high_slope > 0 :         significant = False
    else                                :         significant = True
        
    # ---  T test on the two time intervals ---
    P = sp.stats.ttest_ind(summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]],
                       summary.wofs_ann_pwater.loc[Reference_period[0]:Reference_period[1]]).pvalue.round(4)
    # ----------------------------------------------------    
    
    # ----------------------------------------------------    
    # --- save results to a file ---

    if (verbose) : print('Saving results to : ',rfname)
    # --- write results to a file -- 
    # --- crazy workarounds because sometimes the values are single value arrays and sometimes not!

    if type(reference_pw_area) == type(np.array([1])) : rwa = np.ndarray.item(reference_pw_area) 
    else : rwa = reference_pw_area
    if type(assessment_pw_area) == type(np.array([1])) : awa = np.ndarray.item(assessment_pw_area) 
    else : awa = assessment_pw_area

    water_areas = {}
    with open(rfname, 'r') as f:  
        water_areas = eval(f.read())
    
    if test : print(reference_pw_area,assessment_pw_area,significant,P)
    if not placename in water_areas.keys() : water_areas[placename]={} 
    water_areas[placename][wofs_varname+'_'+str(PWT)+'_'+str(SC)] = (rwa,awa,significant,P)
    with open(rfname, 'w') as f:      f.write(str(water_areas))
    if test: print(water_areas[placename])    
    # ----------------------------------------------------    

    # --- Plot the stuff ---  
    
    if verbose: #plot water variations over time:
        max_water = summary['w95'].max().values
        plt.figure(figsize=[12, 9])
    
        # --- fit some trend curves to the data for visualisation using two-step process, 
        #     viz,. a piecewise linear regression followed by a quadratic interpolator --- 
        
        t_new        = pd.date_range(summary.time.min().values, summary.time.max().values, freq='2M')  
        w80_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w80)[0][1]),kind='quadratic',fill_value='extrapolate')
        w30_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w30)[0][1]),kind='quadratic',fill_value='extrapolate')
        w50_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w50)[0][1]),kind='quadratic',fill_value='extrapolate')
        avf_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.avf)[0][1]),kind='quadratic',fill_value='extrapolate')
        PW_interp    = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.wofs_ann_pwater)[0][1]),kind='quadratic',fill_value='extrapolate')
        PW_max_interp    = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.wofs_ann_pwater_max)[0][1]),kind='quadratic',fill_value='extrapolate')
        PW_min_interp    = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.wofs_ann_pwater_min)[0][1]),kind='quadratic',fill_value='extrapolate')


        # --- actual plots, organised to suit legend  ---
        #plt.plot(t_new,np.clip(w80_interp(t_new),0,max_water),'g',linestyle="dotted")
        #plt.plot(t_new,np.clip(w30_interp(t_new),0,max_water),'r',linestyle="dotted")
        plt.plot(t_new,np.clip(w50_interp(t_new),0,max_water),'b--',marker='')
        plt.plot(t_new,np.clip(avf_interp(t_new),0,max_water),'c',linestyle="dashed")
        plt.plot(summary.time,summary['wofs_ann_pwater'],'k',linestyle="solid", marker="X")  #acutal data points
        plt.plot(reference_times,np.ones(reference_times.size)*reference_pw_area,'r',linestyle="dashed", marker="none")  #acutal data points
        plt.plot(assessment_times,np.ones(assessment_times.size)*assessment_pw_area,'r',linestyle="dashed", marker="none")  #acutal data points

        #plt.plot(t_new,np.clip(PW_interp(t_new),0,max_water) ,'k',linestyle="dashed")
        #plt.plot(t_new,np.clip(PW_max_interp(t_new),0,max_water) ,'k',linestyle="dotted")
        #plt.plot(t_new,np.clip(PW_min_interp(t_new),0,max_water) ,'k',linestyle="dotted")
        #line below is just test code to check on the peicewise linear regression outputs.
        #plt.plot(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w50)[0][1]),'r',linestyle="none",marker="o")
        

        plt.plot(summary.time,summary['w50'],'b',linestyle="none", marker="d")               #acutal data points
        plt.plot(summary.time,summary['avf'],'c',linestyle="none", marker="+")               #acutal data points
        #plt.plot(summary.time,summary['wofs_ann_pwater_max'],'k',linestyle="dotted", marker="none")  #acutal data points
        #plt.plot(summary.time,summary['wofs_ann_pwater_min'],'k',linestyle="dotted", marker="none")  #acutal data points
       
        plt.plot(assessment_times,np.ones(assessment_times.size)*assessment_lq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        plt.plot(assessment_times,np.ones(assessment_times.size)*assessment_uq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        plt.plot(reference_times,np.ones(reference_times.size)*reference_lq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        plt.plot(reference_times,np.ones(reference_times.size)*reference_uq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        
        plt.title(
                placename+\
                "  - Annual Surface Area of Water ("+wofs_varname+")"+"\n Permanent water area change = "\
                 +"("+str(assessment_pw_area)+" - "+str(reference_pw_area)+") / "+str(reference_pw_area)+" = "+str(pw_change_percent)+"%"\
                 +"\n Pixel-wise Permanent Water Threshold (pPWT) = "+str(PW_threshold)+" - "+str(sigma_coefficient)+" Sigma"\
                 +"\n Significant trend based on robust regression over all years = "+str(significant)\
                 +"   T-Test P value = "+str(P)\
                 )
        plt.ylabel("Area (km2)")
        plt.xlabel("Year")
        plt.legend([#'Peak: most (90%) of the year, the detected area of water was less than this (P90 water area)',        
            #'High: most (80%) of the year, the detected area of water was below this line (P80) water area',
            #'Low: some (30%) of the year, the detected area of water was below this line (P30 water area)',
            '50 percentile water area - the area to expect. Half of the time, the water area is greater than this, otherwise less (PWA50)',
            'Average Area of Water (AAW) - can be affected by data gaps and floodplains, etc., which contribute to the average',
            'Area of Permanent Water (PW) - area where freq > : pPWT  (nominally '+str(PWT)+')',
            'Median permanent water area, for Reference and Assessment periods (and interquartile ranges)',
             #'Uncertainty of the AAW (2-sigma) - uncertainty increases with more variable waterbodies and poor data',
#             'Estimate of water area from wet/clear for entire water body'
                   ])
        # --- parameters for the range of the plot --- 
        
        yrange = summary.w90.max() - summary.w30.min()
        ymin   = summary.w30.min() - 0.5  * yrange
        ymin   = summary.wofs_ann_pwater_min.min() * 0.8 
        #ymin   = 3600 
        ymax   = summary.w90.max() + 0.2 * yrange
        if ymin > summary.avf.mean() : 
            ymin   = summary.avf.quantile(0.25)
            yrange = summary.w90.max() - summary.avf.quantile(0.25)
            ymax   = summary.w90.max() + 0.2 * yrange

        # --- manual overrides are sometimes useful
        #ymin = 100 ; ymax = 1500

        plt.ylim(ymin,ymax)
        
        if False : plt.ylim(800,1400)
        
        plt.show()
        if False: 
            plt.savefig('WOfS_SA_'+file_name+'.png',dpi=300)

    #I give the watermask the mean value of the binomial distribution, np. Also need to retain p, for future use
    #could use the sd of the binomial dist, scaled by 1/n, to give an idea of the confidence of p. That would be ((p(1-p))^0.5)/n
    return(summary,ds)

    # --- I think below is redundant --- 
    ds['watermask']  = 1. * ds['wofs_ann_wetcount'] * ds[wofs_varname]  # the float operation avoids wrapping of the integer values
    ds['watermask']  = ds['watermask'].where(ds[wofs_varname] > water_frequency_threshold)  #trims the watermask values

    if test :
        ds['watermask'].where(ds['watermask'] >0).isel(time=7).plot(size=3)
        if False: ds.watermask.plot(col='time')

    return(summary,ds)


In [ ]:
def wofs_summary_old(summary,ds,water_frequency_threshold,
                 wofs_varname='wofs_ann_freq',
                 PWT=0.875,
                 SC=1.2,
                 rfname = 'water_area_results.txt',
                 verbose=True,test=True):
    if verbose:     print('Calculating WOfS summary...')
    if not wofs_varname in ('wofs_ann_freq','wofs_fill_freq') : 
        print ('INVALID VARIABLE NAME, defaulting to wofs_ann_freq')
        wofs_varname = 'wofs_ann_freq'

    # --- produce the following statistics for the dataset and save in the summary dataset:
    # --- the frequency statistics give the numbers, these are conditioned by the methods, which are
    # --- raw, filled, pixel_threshold (r, f, p)
    # --- so the notation is 'W'+str(percentile)+(r,f,p)
    # --- also the average areas of water:
    
    # W_PWT     : uncorrected permanent water area, for the theshold of 'PWT'
    # uc_awa         : uncorrected average water area, for the threhold of
    # 
    # fill_pwa_95     : gap-filled permanent water area, for the theshold of 0.95  
    # fill_pwa_pwt    : gap-filled permanent water area, for the theshold of 'PWT' (this is the 'truth' reference)
    # pwa_pa_PWT      : pixel adjusted permanent water area for the theshold of 'PWT'
    # --- call a separate function? to produce and write these summary values:
    # uc_pwa_PWT_year1_year2   (year1/year2 are the reference period)
    # uc_pwa_PWT_year1_year2   (year1/year2 are the assessment period)
    # uc_awa_year1_year2       (as for above)
    # w10 -> w95       : percentiles as below. W50 is the expected water area. 
    # 
    # --- some parameters ---
    placename       = str(summary.placename.values)
    PW_threshold    = PWT
    sigmacorrection = SC
    
    cell_area = ((ds.x[1]-ds.x[0]) * (ds.y[1]-ds.y[0])) / 1000000
    Reference_period  = ['2000','2019']
    Assessment_period = ['2020','2024']
    reference_times   = summary.time.loc[Reference_period[0]:Reference_period[1]]
    assessment_times   = summary.time.loc[Assessment_period[0]:Assessment_period[1]]
    
    summary['w50'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.50).count(dim=('x','y')) * cell_area)
    summary['w05'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.95).count(dim=('x','y')) * cell_area)
    summary['w95'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.05).count(dim=('x','y')) * cell_area)
    summary['w90'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.10).count(dim=('x','y')) * cell_area)
    summary['w80'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.20).count(dim=('x','y')) * cell_area)
    
    summary['w10'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.90).count(dim=('x','y')) * cell_area)
    summary['w20'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.80).count(dim=('x','y')) * cell_area)
    summary['w30'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.70).count(dim=('x','y')) * cell_area)
    summary['w40'] = (ds[wofs_varname].where(ds[wofs_varname]>=0.60).count(dim=('x','y')) * cell_area)
    summary['avf'] = (ds[wofs_varname].sum(dim=('x','y')) * cell_area) 

    #The last is what is used in the vectorize waterbodies notebook. It is the average area of water. 
    #The average is not a good indicator of what to expect because floodplains, and geography/geometry, skew the distribution.
    #It is also more affected by missing data than, say W50. 
    #In clear-sky areas with floodplains, the average will be higher than w50
    #In cloud-affected areas without floodplains, the average will be below w50
    
    summary['wofs_np'] = (ds.wofs_ann_wetcount * ds.wofs_ann_freq).where(ds.wofs_ann_freq>=0.8).median(dim=('x','y'))
    
    #overall water area - any place observed as water during the observation period.
    water_area = ds.wofs_ann_wetcount.max(dim='time')[:,:].where(ds.wofs_ann_freq.max(dim='time')[:,:] > .0).count() * cell_area
    if test : print('maximum ever water area calculated as : ',water_area,' Total study area is ',ds.x.size * ds.y.size * cell_area)
       
    #--- For each pixel, we can calculate the standard deviation of the frequency based on p; this is pq/N

    #--- issue arises because all-time wofs is not up to date, 
    # ---  in later years there can be data in the annual summaries that are not in the alltime summary, which will throw out results
    # --- work around is to fill the spaces with the p from the annual summary.

    #ds['wofs_ann_p'] = xr.where(~np.isnan(ds.wofs_all_p),ds.wofs_all_p,ds[wofs_varname])
    #ds['wofs_ann_p'] = xr.where(np.isnan(ds.wofs_ann_freq),np.nan,ds.wofs_ann_p)  #use this from here on

    #ds['wofs_ann_freq_sigma'] = ((ds.wofs_ann_p * (1 - ds.wofs_ann_p)) / ds.wofs_ann_clearcount)**0.5
    ds['wofs_ann_freq_sigma'] = ((ds.wofs_ann_freq * (1 - ds.wofs_ann_freq)) / ds.wofs_ann_clearcount)**0.5
    ds['pw_threshold']        = (-1 * ds.wofs_ann_freq_sigma * sigmacorrection) + PW_threshold  #--- threshold varies with p and n

    # --- make a permanent water class
    ds['wofs_ann_permanent'] = xr.where((ds[wofs_varname] - (2 * ds.wofs_ann_freq_sigma)) > ds.pw_threshold,10,
                           xr.where((ds[wofs_varname] + (2 * ds[wofs_varname])) < ds.pw_threshold,1,
                           ds[wofs_varname] * 0 + 5))

    # --- save time series summary --- 
    # --- the min/max values are highly conservative since they assume that all pixels vary in the same direction
    summary['wofs_ann_pwater']     = \
            ds.wofs_ann_permanent.where(ds[wofs_varname]                                > ds.pw_threshold).count(dim=('x','y')) * cell_area
    summary['wofs_ann_pwater_min'] = \
            ds.wofs_ann_permanent.where(ds[wofs_varname] - (ds.wofs_ann_freq_sigma * 2) > PW_threshold).count(dim=('x','y')) * cell_area
    summary['wofs_ann_pwater_max'] = \
            ds.wofs_ann_permanent.where(ds[wofs_varname] + (ds.wofs_ann_freq_sigma * 2) > PW_threshold).count(dim=('x','y')) * cell_area

    # --- the mean frequency is also a measure of the average water area
    summary['wofs_ann_freq_mean'] = ds.wofs_ann_freq.mean(dim=('y','x'))
    summary['wofs_ann_freq_sd'  ] = (ds.wofs_ann_freq.var(dim=('y','x'))**0.5)

    # --- Change statistics:
    reference_pw_area  = summary.wofs_ann_pwater.loc[Reference_period[0]:Reference_period[1]].median(dim='time').values
    assessment_pw_area = summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]].median(dim='time').values
    pw_change_percent = max(abs((100 * (assessment_pw_area / reference_pw_area - 1.0))),
                            0.000000000000001) # --- to prevent problems with infinities if change is really zero--#

    if test : print(reference_pw_area,assessment_pw_area,pw_change_percent)
    
    # --- round these values:
    if reference_pw_area > 0.0001 :
        reference_pw_area  = np.round(reference_pw_area,3-int((np.log10(reference_pw_area))))
        assessment_pw_area  = np.round(assessment_pw_area,3-int((np.log10(assessment_pw_area))))
        pw_change_percent  = np.round(pw_change_percent,3-int((np.log10(abs(pw_change_percent)))))
    else:
        reference_pwater_area = np.round(0.0)
    if not assessment_pw_area > 0.00001: assessment_pw_area = np.round(0.0)
    if test : 
        print('types:',type(assessment_pw_area),type(reference_pw_area))
    
    
    assessment_uq_area = summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]].quantile(.75,dim='time').values
    assessment_lq_area = summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]].quantile(.25,dim='time').values
    reference_uq_area  = summary.wofs_ann_pwater.loc[Reference_period[0] :Reference_period[1]] .quantile(.75,dim='time').values
    reference_lq_area  = summary.wofs_ann_pwater.loc[Reference_period[0] :Reference_period[1]] .quantile(.25,dim='time').values
    
    # ----------------------------------------------------    
    # --- Robust regression on the annual areas for the reference and assessment period ---
    #slope,intercept,low_slope,high_slope = sp.stats.theilslopes(y,x,method='joint')
    x = summary.time 
    y = summary.wofs_ann_pwater 
    slope,intercept,low_slope,high_slope = sp.stats.theilslopes(y,x,method='joint')
    
    if low_slope < 0 and high_slope > 0 :         significant = False
    else                                :         significant = True
        
    # ---  T test on the two time intervals ---
    P = sp.stats.ttest_ind(summary.wofs_ann_pwater.loc[Assessment_period[0]:Assessment_period[1]],
                       summary.wofs_ann_pwater.loc[Reference_period[0]:Reference_period[1]]).pvalue.round(4)
    # ----------------------------------------------------    
    
    # ----------------------------------------------------    
    # --- save results to a file ---

    if (verbose) : print('Saving results to : ',rfname)
    # --- write results to a file -- 
    # --- crazy workarounds because sometimes the values are single value arrays and sometimes not!

    if type(reference_pw_area) == type(np.array([1])) : rwa = np.ndarray.item(reference_pw_area) 
    else : rwa = reference_pw_area
    if type(assessment_pw_area) == type(np.array([1])) : awa = np.ndarray.item(assessment_pw_area) 
    else : awa = assessment_pw_area

    water_areas = {}
    with open(rfname, 'r') as f:  
        water_areas = eval(f.read())
    
    if test : print(reference_pw_area,assessment_pw_area,significant,P)
    if not placename in water_areas.keys() : water_areas[placename]={} 
    water_areas[placename][wofs_varname+'_'+str(PWT)+'_'+str(SC)] = (rwa,awa,significant,P)
    with open(rfname, 'w') as f:      f.write(str(water_areas))
    if test: print(water_areas[placename])    
    # ----------------------------------------------------    

    # --- Plot the stuff ---  
    
    if verbose: #plot water variations over time:
        max_water = summary['w95'].max().values
        plt.figure(figsize=[12, 9])
    
        # --- fit some trend curves to the data for visualisation using two-step process, 
        #     viz,. a piecewise linear regression followed by a quadratic interpolator --- 
        
        t_new        = pd.date_range(summary.time.min().values, summary.time.max().values, freq='2M')  
        w80_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w80)[0][1]),kind='quadratic',fill_value='extrapolate')
        w30_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w30)[0][1]),kind='quadratic',fill_value='extrapolate')
        w50_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w50)[0][1]),kind='quadratic',fill_value='extrapolate')
        avf_interp   = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.avf)[0][1]),kind='quadratic',fill_value='extrapolate')
        PW_interp    = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.wofs_ann_pwater)[0][1]),kind='quadratic',fill_value='extrapolate')
        PW_max_interp    = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.wofs_ann_pwater_max)[0][1]),kind='quadratic',fill_value='extrapolate')
        PW_min_interp    = sp.interpolate.interp1d(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.wofs_ann_pwater_min)[0][1]),kind='quadratic',fill_value='extrapolate')


        # --- actual plots, organised to suit legend  ---
        #plt.plot(t_new,np.clip(w80_interp(t_new),0,max_water),'g',linestyle="dotted")
        #plt.plot(t_new,np.clip(w30_interp(t_new),0,max_water),'r',linestyle="dotted")
        plt.plot(t_new,np.clip(w50_interp(t_new),0,max_water),'b--',marker='')
        plt.plot(t_new,np.clip(avf_interp(t_new),0,max_water),'c',linestyle="dashed")
        plt.plot(summary.time,summary['wofs_ann_pwater'],'k',linestyle="solid", marker="X")  #acutal data points
        plt.plot(reference_times,np.ones(reference_times.size)*reference_pw_area,'r',linestyle="dashed", marker="none")  #acutal data points
        plt.plot(assessment_times,np.ones(assessment_times.size)*assessment_pw_area,'r',linestyle="dashed", marker="none")  #acutal data points

        #plt.plot(t_new,np.clip(PW_interp(t_new),0,max_water) ,'k',linestyle="dashed")
        #plt.plot(t_new,np.clip(PW_max_interp(t_new),0,max_water) ,'k',linestyle="dotted")
        #plt.plot(t_new,np.clip(PW_min_interp(t_new),0,max_water) ,'k',linestyle="dotted")
        #line below is just test code to check on the peicewise linear regression outputs.
        #plt.plot(summary.time,(piecewise_linreg(np.arange(0,summary.time.size),summary.w50)[0][1]),'r',linestyle="none",marker="o")
        

        plt.plot(summary.time,summary['w50'],'b',linestyle="none", marker="d")               #acutal data points
        plt.plot(summary.time,summary['avf'],'c',linestyle="none", marker="+")               #acutal data points
        #plt.plot(summary.time,summary['wofs_ann_pwater_max'],'k',linestyle="dotted", marker="none")  #acutal data points
        #plt.plot(summary.time,summary['wofs_ann_pwater_min'],'k',linestyle="dotted", marker="none")  #acutal data points
       
        plt.plot(assessment_times,np.ones(assessment_times.size)*assessment_lq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        plt.plot(assessment_times,np.ones(assessment_times.size)*assessment_uq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        plt.plot(reference_times,np.ones(reference_times.size)*reference_lq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        plt.plot(reference_times,np.ones(reference_times.size)*reference_uq_area,'k',linestyle="dotted", marker="none")  #acutal data points
        
        plt.title(
                placename+\
                "  - Annual Surface Area of Water ("+wofs_varname+")"+"\n Permanent water area change = "\
                 +"("+str(assessment_pw_area)+" - "+str(reference_pw_area)+") / "+str(reference_pw_area)+" = "+str(pw_change_percent)+"%"\
                 +"\n Pixel-wise Permanent Water Threshold (pPWT) = "+str(PW_threshold)+" - "+str(sigmacorrection)+" Sigma"\
                 +"\n Significant trend based on robust regression over all years = "+str(significant)\
                 +"   T-Test P value = "+str(P)\
                 )
        plt.ylabel("Area (km2)")
        plt.xlabel("Year")
        plt.legend([#'Peak: most (90%) of the year, the detected area of water was less than this (P90 water area)',        
            #'High: most (80%) of the year, the detected area of water was below this line (P80) water area',
            #'Low: some (30%) of the year, the detected area of water was below this line (P30 water area)',
            '50 percentile water area - the area to expect. Half of the time, the water area is greater than this, otherwise less (PWA50)',
            'Average Area of Water (AAW) - can be affected by data gaps and floodplains, etc., which contribute to the average',
            'Area of Permanent Water (PW) - area where freq > : pPWT  (nominally '+str(PWT)+')',
            'Median permanent water area, for Reference and Assessment periods (and interquartile ranges)',
             #'Uncertainty of the AAW (2-sigma) - uncertainty increases with more variable waterbodies and poor data',
#             'Estimate of water area from wet/clear for entire water body'
                   ])
        # --- parameters for the range of the plot --- 
        
        yrange = summary.w90.max() - summary.w30.min()
        ymin   = summary.w30.min() - 0.5  * yrange
        ymin   = summary.wofs_ann_pwater_min.min() * 0.8 
        #ymin   = 3600 
        ymax   = summary.w90.max() + 0.2 * yrange
        if ymin > summary.avf.mean() : 
            ymin   = summary.avf.quantile(0.25)
            yrange = summary.w90.max() - summary.avf.quantile(0.25)
            ymax   = summary.w90.max() + 0.2 * yrange

        # --- manual overrides are sometimes useful
        #ymin = 100 ; ymax = 1500

        plt.ylim(ymin,ymax)
        
        if False : plt.ylim(800,1400)
        
        plt.show()
        if False: 
            plt.savefig('WOfS_SA_'+file_name+'.png',dpi=300)

    #I give the watermask the mean value of the binomial distribution, np. Also need to retain p, for future use
    #could use the sd of the binomial dist, scaled by 1/n, to give an idea of the confidence of p. That would be ((p(1-p))^0.5)/n
    return(summary,ds)

    # --- I think below is redundant --- 
    ds['watermask']  = 1. * ds['wofs_ann_wetcount'] * ds[wofs_varname]  # the float operation avoids wrapping of the integer values
    ds['watermask']  = ds['watermask'].where(ds[wofs_varname] > water_frequency_threshold)  #trims the watermask values

    if test :
        ds['watermask'].where(ds['watermask'] >0).isel(time=7).plot(size=3)
        if False: ds.watermask.plot(col='time')

    return(summary,ds)


In [ ]:
#This code fills one list so that the values are ascending with another list.

#for EXAMPLE, the second list is wofs_all_freq, the first list wofs_ann_freq for some year. 
#Should be applied only where water is normally expected... eg 70%. Otherwise, a single outlier of water can trigger filling of, e.g., floodplains.

'''
Although the algorithm is only order n, it is insanely slow; so there is probably some innate inefficiency.

Algorithm will misperform on waterbodies that have seen signficant siltation through time, for example, because the reference histogrram is no longer valid.
there is a dam in morocco where this happens
To mitigate this a threshold of change is needed based on teh confidence that there is in each pixel value, which can be indicated with sigma2 = pq/n
To accomodate this an additional array is included in the arguments 
A further mitigation would be to base the reference values on the past 10 years of data, rather than the full time series
A further moderation on behaviour would be to only promote values where the 'wetter' pixel has N >= the less wet pixel

'''

def wofs_gapfill(var,ref_var,var_sigma,verbose=True,test=True,freq_low_threshold = 0.8) :
    verbose = verbose or test
    #convert the input arrays, assumed to be 2d, to 1-d, replace nans with zeros, force a copy, and stack them
    var = var.copy()  ; ref_var = ref_var.copy()
    sigma_factor = 0.5    #no value will be changed by more than 0.5 sigma from the observed value
    var = np.reshape(var,(var.size))                 ; var = np.where(np.isnan(var),0.,var)
    ref = np.reshape(ref_var,(ref_var.size))         ; ref = np.where(np.isnan(ref),0.,ref)
    var_sigma = np.reshape(var_sigma,(ref_var.size)) ; var_sigma = np.where(np.isnan(var_sigma),0.,var_sigma)
    ind = np.arange(0.,ref_var.size)     #index will be used to recover the original order
    stack = np.vstack((ind,var,ref,var_sigma))    #stack
    sort_index = stack.argsort(1)[2]              #sorting along rows taking row #2

    #make a new, sorted, copy:
    new = np.zeros(stack.size,dtype='float32').reshape(stack.shape)
    for i in range(0,len(sort_index)) :
        new[:,i] = stack[:,sort_index[i]] #the new data are sorted on the reference variable
        
    #manipulate the target variable to conform with the reference variable - values increasing with the sort
    
    for i in range(0,len(sort_index)-1) :   
        #print(i,new[1,i],new[1,i+1],new[1,i]>new[1,i+1])
        if     (new[1,i]   > (new[1,i+1] + (sigma_factor * new[3,i+1])) and new[2,i+1] > freq_low_threshold ) : 
            new [1,i+1] = new[1,i]

    #return the array to original sort order 
    resorted = np.copy(new)*0
    for i in range(0,len(sort_index)) :   
        resorted[:,new[0,i].astype(int)] = new[:,i]    
        
    return resorted[1,:].reshape(ref_var.shape)     #return target variable in good shape
    return resorted[1,:].reshape(ref_var.shape),new[2,:].resahpe(ref_var.shape)   #return target variable and reference histrogram in good shape
    return resorted[3,:].argsort(0).reshape(ref_var.shape)     #return the sort index in correct shape
        

In [5]:
def plot_wofs_gapfill(ds):   #PLOTS the cumulative histogram of the gap-filled wofs, per year.
    # --- 
    ncells = ds.wofs_all_freq.size
    #ref    = ds.wofs_all_freq.values.reshape(ncells)
    #for dams built since the start of wofs, the wofs_all variable underestimates
    ref    = np.maximum(ds.wofs_all_freq,ds.wofs_ann_freq.mean(dim='time')).values.reshape(ncells)
    

    for i in range(0,ds.time.size,3):
        year = ds.time[i]
        ann    = ds.wofs_ann_freq.sel(time=year)[:,:].values.reshape(ncells)
        fill   = ds.wofs_fill_freq.sel(time=year)[:,:].values.reshape(ncells)
        stack = np.vstack((ref,ann,fill))    #stack
        sort_index = stack.argsort(1)[0]    #soring along rows, and taking the first row

        #make a new, sorted, copy:
        new = np.zeros(stack.size,dtype='float32').reshape(stack.shape)
        for j in range(0,len(sort_index)) :
            new[:,j] = stack[:,sort_index[j]]    #the new data are sorted on the reference variable
            if np.isnan(new[0,j]) or np.isnan(new[1,j]) : new[2,j] = np.nan  #avoid nans reverting to zero

        plt.figure(figsize=(10,5))
        plt.subplot(131); plt.plot(new[0][0:]); plt.title('Reference \n- rising frequency curve\n n='+str(ncells))
        plt.subplot(132); plt.plot(new[1][0:]); plt.title('Original, \nnoisy data\nyear = '+str(ds.time[i].values)[0:4])
        plt.subplot(133); plt.plot(new[2][0:]); plt.title('Filled data \n- rising frequency curve,\n-at high freq values')
        plt.show()



In [27]:
def display_rgb_dataset(ds,bands,timeselection=False,test=True,verbose=True):
    if timeselection:
        start=timeselection[0]; end=timeselection[1];steps=timeselection[2]
    else:
        start  = 19;    
        steps   = 3;     
        end     = np.size(ds.time)-1
    interval = int((end-start)/steps)
    year_index =ds.time[start:end:interval]

    rgbdataset  = xr.merge([ds[bands[0]].loc[year_index],\
                 ds[bands[1]].loc[year_index],\
                 ds[bands[2]].loc[year_index]])
    
    rgb(rgbdataset,bands=bands,col="time")
    
    
    

### Colour space transformation on MSI data.
- Method is as per Van Der Woerd 2018.
- More accurate hue angles are retrieved if more bands are used;
- results for ETM+ are therefore  less accurate than for MSI and OLI
- Cannot run for OLI at this point, because we don't have band1 in the geomedian
- Results should in general be more accurate with more bands, but MSI, OLI and ETM are limited


#### Steps in the calculation:
- Calculate  X, Y, and Z, each of which is a linear combination of certain bands.
- Convert to chromaticity coordinates, x,y using:
- x = X/(X + Y + Z), and y= Y/(X + Y + Z)
- in this space, white is when x = y = 1/3. Call that xw, yw 
- convert to polar coordinates with the white point at the origin, to give a hue angle which increases anti-clockwise from y - yw = 0. 
- (the hue angle is the angle made by the vector [x-xw, y-yw] with the x axis)
- apply a correction factor, to correct for the systematic variations from the correct hue angle.

##### Whilst I have done this per pixel, one could average the reflectance values for the area and calculate the hue from that.

#### Coefficients - specific to each sensor - to conver to chromatic space 

S2 MSI-20 m 

| (nm) | R400  | R490   | R560   | R665   | R705  | R710 |
|------|-------|--------|--------|--------|-------|------|
| Band |       |  2     |   3    |   4    |   5   |      |  
|  X   | 8.356 | 12.040 | 53.696 | 32.028 | 0.529 | 0.016 |
|  Y   | 0.993 | 23.122 | 65.702 | 16.808 | 0.192 | 0.006 |
|  Z   | 43.487| 61.055 | 1.778  | 0.015  | 0.000 | 0.000 |


OLI:

| (nm) | R400 | R443 | R482 | R561 | R655 | R710 |
|------|------|------|------|------|------|------|
| Band |      |  1   |   2  |   3  |   4  |      |  
| X | 2.217 | 11.053 | 6.950 | 51.135 | 34.457 | 0.852
| Y | 0.082 | 1.320 | 21.053 | 66.023 | 18.034 | 0.311
| Z | 10.745 | 58.038 | 34.931 | 2.606 | 0.016 | 0.000 

ETM 

|(nm) | R400   |  R485  | R565   | R660   | R710
|-----|--------|--------|--------|--------|--------|
|Band |        |   1    |  2     |   3    |      |
| X   | 7.8195 | 13.104 | 53.791 | 31.304 | 0.6463
| Y   | 0.807  | 24.097 | 65.801 | 15.883 | 0.235
| Z   | 40.336 | 63.845 | 2.142  | 0.013  | 0.000

#### Coefficients to calculate the hue angle offset
a = hue_angle/100
delta is the sum of the fith order polynomial formed from the powers of a and the relevant coefficients (i.e., for the sensor)
delta is in degrees, and is added to a to give the corrected hue angle. 


| Sensor | Resolution | a5 | a4 | a3 | a2 | a | offset |
|--------|------------|----|----|----|----|---|--------|
|S2 MSI  | 20         | -161.23  | 1117.08  | -2950.14  | 3612.17  | -1943.57  | 364.28
Landsat 8 OLI         | 30  | -52.16  | 373.81  |  -981.83  | 1134.19  | -533.61  | 76.72
Landsat 7 ETM+        | 30  | -84.94  | 594.17  | -1559.86  | 1852.50  | -918.11  | 151.49

In [24]:
# hue adjustment coefficients for MSI. This is the final step but its a function called by the second function..
# I am sure that there are more efficient ways to code this as a matrix multiplication... 

def hue_adjust_old(dataset) :
        # ---- this function makes an adjustment to the hue to produce the final value ----
        deltahuemsi = (-161.23  , 1117.08 , -2950.14  , 3612.17  , -1943.57  , 364.28)
        deltahueoli = ( -52.16  , 373.81  ,  -981.83  , 1134.19  ,  -533.61  , 76.72)
        deltahueetm = ( -84.94  , 594.17  , -1559.86  , 1852.50  ,  -918.11  , 151.49)
        dataset['hue_delta'] = (dataset['hue']/100)**5 * deltahuemsi[0]  +   \
               (dataset['hue']/100)**4 *deltahuemsi[1] + \
               (dataset['hue']/100)**3 *deltahuemsi[2] + \
               (dataset['hue']/100)**2 *deltahuemsi[3] + \
               (dataset['hue']/100)**1 *deltahuemsi[4] + \
               (dataset['hue']/100)**0 *deltahuemsi[5]
        dataset['hue'] = dataset['hue'] + dataset['hue_delta']
        dataset = dataset.drop_vars('hue_delta')
        return(dataset)


def hue_calculation_old(dataset,instrument='msi_agm',test=False,verbose=False) : 
    #---- the hue is calculated by conversion of the wavelengths to chromatic coordinates using sensor-specific coefficients
    
    #enter the x,y,z msi chromatic coefficients...
    chrom_coeffs = {
        'X': {'msi01':8.356, 'msi02':12.040,'msi03': 53.696,'msi04':32.028,'msi05': 0.529}, #x msi chromaticity
        'Y': {'msi01':0.993, 'msi02':23.122,'msi03': 65.702,'msi04':16.808,'msi05': 0.192},
        'Z': {'msi01':43.487,'msi02':61.055,'msi03':  1.778,'msi04': 0.015,'msi05': 0.000},
        }
    #use all the bands that are in the dataset (band1 is probably missing..):
    if instrument == 'msi_agm' : band_list = ['msi01_agmr','msi02_agmr','msi03_agmr','msi04_agmr','msi05_agmr']
    if instrument == 'msi'     : band_list = ['msi01r'    ,'msi02r'    ,'msi03r'    ,'msi04r'    ,'msi05r']
    
    var_list = []
    for name in band_list:
        if name in dataset.data_vars:
            var_list = np.append(var_list,name)

    #initiate two Datasets with no variables:
    Cdata         = xr.zeros_like(dataset).drop_vars(dataset.data_vars)
    Cdata_summary       = dataset.drop_dims(['x','y'])  #summary is not required in the per-pixel processing
    #Cdata_summary = xr.zeros_like(summary).drop_vars(summary.data_vars) # this is not used in the per-pixel processing
    
    n = 1 ; s = np.array([], dtype=np.int8)
    for d in Cdata.dims:        n = n * Cdata[d].size;         s = np.append(s,Cdata[d].size)

    for XYZ in chrom_coeffs.keys() :
        Cdata[XYZ] = Cdata.dims, np.zeros(n).reshape(s)                #-- initiate a dataset
        for var in var_list:
            var_shortname = var[0:5]   #-- drops the 'r', or '_agmr' from the variable name
            Cdata[XYZ] = Cdata[XYZ] + ds[var] * chrom_coeffs[XYZ][var_shortname]
            
    # ---- normalise the X and Y parameters
    Cdata["Xn"] = Cdata['X'] / ( Cdata['X'] +  Cdata['Y'] +  Cdata['Z'])
    Cdata["Yn"] = Cdata['Y'] / ( Cdata['X'] +  Cdata['Y'] +  Cdata['Z'])
    Xwhite =  Ywhite = 1 / 3.
    # ---- calculate the delta to white ----
    Cdata['Xnd'] = Cdata['Xn'] - Xwhite; 
    Cdata['Ynd'] = Cdata['Yn'] - Ywhite; 

    # ---- convert vector to angle ----
    Cdata['hue'] = np.mod(np.arctan2(Cdata['Ynd'],Cdata['Xnd'])*(180.00/np.pi) +360.,360)  

    # ---- this gives the correct mathematical angle, ie. from 0 (=east), counter-clockwise as a positive number
    # ---- note the 'arctan2' function, and that x and y are switched compared to expectations

    # ---- code below is not used for pixel level processing, but is used by others / later!
    if False:
        Cdata_summary['Xnd'] = Cdata['Xnd'].where(dataset['wofs_ann_freq']>0.9).median(dim=('x','y'))
        Cdata_summary['Ynd'] = Cdata['Ynd'].where(dataset['wofs_ann_freq']>0.9).median(dim=('x','y'))  
        Cdata_summary['hue'] = np.mod(np.arctan2(Cdata_summary['Ynd'],Cdata_summary['Xnd'])*(180.00/np.pi) +360.,360)  

    #apply the hue adjustment - only do it once!
    if test: print('Average Hue values pre-adjustment :', Cdata_summary['hue'].values.round(1)) 
    Cdata         = hue_adjust(Cdata)
    Cdata_summary = hue_adjust(Cdata_summary) 
    if test: print('Average Hue values post-ajustment :', Cdata_summary['hue'].values.round(1)) 

    #plot the vectors - dev code only
    if verbose:
        print('Average Hue values post-ajustment :', Cdata_summary['hue'].values.round(1)) 
        if test:
            print('Hue vectors :', Cdata['Ynd'].mean(dim=('x','y')).values.round(4),Cdata['Ynd'].median(dim=('x','y')).values.round(4) )
            print('Hue vectors :', Cdata['Xnd'].mean(dim=('x','y')).values.round(4),Cdata['Xnd'].median(dim=('x','y')).values.round(4) )
        if test:Cdata['Ynd'].sel(time=slice('2019','2020')).plot(col='time',robust=True,vmin=0)#,vmax=360,cmap='hsv')
    return Cdata['hue'],Cdata_summary['hue']  #the second output is not required for pixel-level processing, but is used by some of my code. 
   

In [44]:
#helpful function when renaming variables
def rename_vars_robust_old(dataset,var_names,verbose=False):
        for var in var_names :
            if var[0] in np.asarray(dataset.data_vars):
                dataset = dataset.rename_vars({var[0]:var[1]})
        if verbose: 
            print(dataset.data_vars)
        return(dataset)

In [ ]:
#dark pixel correction over water based on the assumption that longer wavelengths
#should be zero if not for atmospheric scattering
'''
- Identify min values of longer (SWIR) wavelengths (bands: msi12; oli07)
- Calculate the appropriate adjustment for other band intervals. 
(for now, simply take 1* SR(IR)) (the simplest approach is to take the value itself)
- Subtract values whilst preserving mmt ge zero.
- _Sometimes_, this helps!
'''

#--- this coded has been moved to _WQ_functions.py

def R_correction_old(ds,instruments,water_frequency_threhold,verbose=False,test=True):
    #for each variable in the list, reduce by the value of the ref_variable, over target areas
    #target areas are areas where the frequency of water is ge than the water frequency threshold parameter
    ### Rayleigh correction - dark pixel adjustment

    dp_adjust = { 
        'msi_agm': {'ref_var':'msi12_agm','var_list': ['msi04_agm','msi03_agm','msi02_agm','msi05_agm','msi06_agm', 'msi07_agm']},
        'oli_agm': {'ref_var':'oli07_agm','var_list': ['oli04_agm','oli03_agm','oli02_agm']},
        'tm_agm' : {'ref_var':'tm07_agm','var_list': ['tm04_agm','tm03_agm','tm02_agm','tm01_agm']}
        }
    
    for sensor in dp_adjust.keys():
        #if instruments_to_use[sensor]['use']:        
        if sensor in list(instruments.keys()):        
            ref_var = dp_adjust[sensor]['ref_var']
            if not ref_var in ds.data_vars :
                print('variable',ref_var,'expected  but not found in the dataset - correction FAILING')
                return(ds)
            else:
                for target_var in dp_adjust[sensor]['var_list']:
                    if test: print(target_var)
                    if not target_var in ds.data_vars :
                        print('variable',target_var,'expected  but not found in the dataset')
                        return(ds)
                    else:
                        if test : print('calculating values for :',str(target_var+'r'))
                        new_var = str(target_var+'r')
                        #ds[new_var]  = (ds[target_var] - ds[ref_var]).where(ds[target_var] > ds[ref_var],0).where(ds[target_var]>0)
                        ds[new_var] = (ds[target_var] - ds[ref_var]).\
                            where(ds[target_var] > ds[ref_var],0).\
                            where(ds[target_var]>0).\
                            where(ds.wofs_fill_freq>water_frequency_threshold,ds[target_var])

    if verbose : #plot graphs of TM illustrating the shift in the distribution
        y = np.random.rand(np.size(ds.msi02_agm))  #creates a random field with the right number of values..
        target_n = 10000
        if np.size(y) > target_n: rand_cut = target_n/np.size(y)
        else: rand_cut = 1
        effective_n = np.size(y[np.argwhere(y<rand_cut)])
        
        if True:
            x = np.arange(0,effective_n)
            plt.plot(np.sort(ds['msi02_agmr'].values.reshape(np.size(ds['msi02_agmr'])))[np.argwhere(y<rand_cut)],x,"b-")
            plt.plot(np.sort(ds['msi02_agm' ].values.reshape(np.size(ds['msi02_agm' ])))[np.argwhere(y<rand_cut)],x,"b--" )
            plt.plot(np.sort(ds['msi03_agmr'].values.reshape(np.size(ds['msi02_agm'])))[np.argwhere(y<rand_cut)],x,"g-")
            plt.plot(np.sort(ds['msi03_agm'].values.reshape(np.size(ds['msi02_agm'])))[np.argwhere(y<rand_cut)],x,"g--")
            plt.plot(np.sort(ds['msi04_agmr'].values.reshape(np.size(ds['msi02_agm'])))[np.argwhere(y<rand_cut)],x,"r-")
            plt.plot(np.sort(ds['msi04_agm'].values.reshape(np.size(ds['msi02_agm'])))[np.argwhere(y<rand_cut)],x,"r--",label='what the ...')
            
            #plt.plot(ds['etm05'].values.reshape(np.size(ds['etm01'])),ds['etm01'].values.reshape(np.size(ds['etm01'])),"o"    
    print('R_correction completed normally')
    return(ds)
    

### Sentinel to Landsat linear correction for water areas (not used at this point)

- an OLS linear regression between the matching bands is what is generally used
##### Option A : use HLS or Flood corrections
- general parameters are available (eg flood, nasa), as below
##### Option B : Derive a local regresson using np.polyfit
- Ideally we might determine a pan-african correction for water areas only.